# Differential Equations — Session 23
## Section 5.1: Linear Models — Initial-Value Problems

**Planned length:** 90 minutes  
**Notebook type:** Student interactive lecture

### Learning objectives

Students should be able to:

1. derive a spring–mass model from Hooke's law and Newton's second law;
2. solve and interpret free undamped motion;
3. convert between coefficient form and amplitude–phase form;
4. derive effective stiffness for springs in parallel and in series;
5. classify damped motion as over-, critically, or underdamped;
6. distinguish transient and steady-state responses;
7. compute and interpret the frequency response of a forced oscillator;
8. translate the mechanical model to an LRC-series circuit.

> The notebook preserves the theoretical structure of Section 5.1 while using original examples, diagrams, and simulations.

**Edition:** Local Instructor Interactive Edition

> **Student interactive edition — local Jupyter/Cursor workflow**
>
> 1. Run the **Local notebook setup** cell below first.
> 2. Read each explanation and derivation in order.
> 3. Run simulation and visualization cells as you reach them.
> 4. At each **Classroom Checkpoint**, stop and work out your answer before class discussion continues.
> 5. This student edition intentionally contains **no instructor answer-reveal cells and no instructor solution notes**.
>
> **Tip:** During class, use `Shift + Enter` to move through the notebook one cell at a time.

In [ ]:
# Local notebook setup — run this cell first.
import importlib.util
import platform
import sys

_REQUIRED = ["numpy", "matplotlib", "scipy", "sympy", "ipywidgets"]
_missing = [name for name in _REQUIRED if importlib.util.find_spec(name) is None]

print(f"Python {sys.version.split()[0]} on {platform.system()}")
if _missing:
    print("Missing packages:", ", ".join(_missing))
    print("From the project folder, run:")
    print("python -m pip install -r requirements.txt")
else:
    print("Student notebook environment is ready.")

### Core 90-minute path

| Time | Topic |
|---:|---|
| 0–18 min | Hooke's law, equilibrium, and the undamped model |
| 18–35 min | Equation of motion, amplitude, phase, and energy |
| 35–58 min | Damping classification |
| 58–78 min | Periodic forcing, transient/steady state, frequency response |
| 78–88 min | Mechanical–electrical analogy |
| 88–90 min | Exit check |

Double springs, time-varying stiffness, and detailed LRC calculations are marked as extensions.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from scipy.integrate import solve_ivp, quad
from scipy.optimize import brentq
from IPython.display import display, Markdown

try:
    from ipywidgets import interact, FloatSlider, IntSlider, Dropdown
    WIDGETS_AVAILABLE = True
except ImportError:
    WIDGETS_AVAILABLE = False

plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['axes.grid'] = True
np.set_printoptions(precision=6, suppress=True)

def solve_second_order(accel, t_span, y0, points=1000, **kwargs):
    def rhs(t, z):
        x, v = z
        return [v, accel(t, x, v)]
    t_eval = np.linspace(t_span[0], t_span[1], points)
    return solve_ivp(rhs, t_span, y0, t_eval=t_eval, **kwargs)

print('Notebook ready.')
print('Interactive widgets available:', WIDGETS_AVAILABLE)

## Formal theory reference

### Definition 5.1-A — Hooke's law

For an ideal linear spring, the restoring force is

$$
F_s=-kx,
$$

where $x$ is displacement from equilibrium and $k>0$ is the spring stiffness.

### Principle 5.1-B — Equilibrium shift

For a vertical spring, gravity determines the static stretch $s$ through

$$
mg=ks.
$$

When dynamic displacement $x(t)$ is measured from the equilibrium position, the constant gravitational terms cancel.

### Theorem 5.1-C — Free undamped motion

The IVP

$$
mx''+kx=0,
\qquad
x(0)=x_0,
\qquad
x'(0)=v_0
$$

has the solution

$$
x(t)=x_0\cos(\omega_0t)+\frac{v_0}{\omega_0}\sin(\omega_0t),
\qquad
\omega_0=\sqrt{\frac{k}{m}}.
$$

The period and frequency are

$$
T=\frac{2\pi}{\omega_0},
\qquad
f=\frac{\omega_0}{2\pi}.
$$

### Proposition 5.1-D — Amplitude–phase form

Every function

$$
x=c_1\cos(\omega t)+c_2\sin(\omega t)
$$

can be written as

$$
x=A\cos(\omega t-\delta),
$$

where

$$
A=\sqrt{c_1^2+c_2^2},
\qquad
\delta=\operatorname{atan2}(c_2,c_1).
$$

### Proposition 5.1-E — Mechanical energy

For free undamped motion,

$$
E=\frac12m(x')^2+\frac12kx^2
$$

is constant.

### Theorem 5.1-F — Damping classification

For

$$
mx''+cx'+kx=0,
$$

let

$$
\Delta=c^2-4mk.
$$

- $\Delta>0$: overdamped;
- $\Delta=0$: critically damped;
- $\Delta<0$: underdamped.

For $c>0$, every free response tends to zero as $t\to\infty$.

### Theorem 5.1-G — Sinusoidal steady-state amplitude

For

$$
mx''+cx'+kx=F_0\cos(\gamma t),
$$

the steady-state displacement has amplitude

$$
A(\gamma)=
\frac{F_0}
{\sqrt{(k-m\gamma^2)^2+(c\gamma)^2}}.
$$

### Model 5.1-H — LRC-series circuit

Charge $q(t)$ in an LRC circuit satisfies

$$
Lq''+Rq'+\frac1Cq=E(t),
$$

with current $i=q'$. The mechanical correspondence is

$$
m\leftrightarrow L,
\qquad
c\leftrightarrow R,
\qquad
k\leftrightarrow \frac1C,
\qquad
F(t)\leftrightarrow E(t).
$$

### Classroom Checkpoint — Damping Classification

For

$$
my''+cy'+ky=0,
$$

which discriminant distinguishes underdamped, critically damped, and overdamped motion?

> Pause here. Before continuing, try to explain their reasoning before continuing.

## 1. From physical forces to the differential equation

Take downward displacement from equilibrium as positive. The spring extension is $s+x$, so the forces are

$$
F_s=-k(s+x),
\qquad
F_g=mg.
$$

Newton's law gives

$$
mx''=-k(s+x)+mg.
$$

Using $mg=ks$ reduces this to

$$
mx''+kx=0.
$$

The equilibrium coordinate is the reason gravity disappears from the dynamic equation.

In [ ]:
# Original schematic of a vertical spring-mass system
fig, ax = plt.subplots(figsize=(5, 6))
ax.axis('off')
ax.plot([0.25, 0.75], [0.92, 0.92], linewidth=3)
# spring zig-zag
xs = np.linspace(0.5, 0.5, 2)
y_top, y_bottom = 0.90, 0.50
zig_y = np.linspace(y_top, y_bottom, 17)
zig_x = 0.5 + 0.06*np.array([0,1,-1,1,-1,1,-1,1,-1,1,-1,1,-1,1,-1,1,0])
ax.plot(zig_x, zig_y, linewidth=2)
rect = plt.Rectangle((0.39, 0.35), 0.22, 0.15, fill=False, linewidth=2)
ax.add_patch(rect)
ax.text(0.50, 0.425, 'mass $m$', ha='center', va='center')
ax.axhline(0.28, linestyle='--')
ax.text(0.70, 0.28, 'equilibrium $x=0$', va='center')
ax.annotate('', xy=(0.82,0.15), xytext=(0.82,0.28), arrowprops={'arrowstyle':'->'})
ax.text(0.87,0.20,'positive $x$', rotation=90, va='center')
ax.set_xlim(0,1); ax.set_ylim(0,1)
plt.show()

## 2. Free undamped motion

Consider

$$
2x''+18x=0,
\qquad
x(0)=0.4,
\qquad
x'(0)=-0.9.
$$

Here

$$
\omega_0=3,
$$

so

$$
x(t)=0.4\cos(3t)-0.3\sin(3t).
$$

The amplitude is $0.5$, even though neither coefficient equals $0.5$.

In [ ]:
m, k = 2.0, 18.0
omega = np.sqrt(k/m)
x0, v0 = 0.4, -0.9
c1, c2 = x0, v0/omega
A = np.hypot(c1, c2)
delta = np.arctan2(c2, c1)

print('c1 =', c1, 'c2 =', c2)
print('amplitude =', A)
print('phase delta =', delta)
print('period =', 2*np.pi/omega)

t = np.linspace(0, 4*2*np.pi/omega, 800)
x = c1*np.cos(omega*t)+c2*np.sin(omega*t)
v = -c1*omega*np.sin(omega*t)+c2*omega*np.cos(omega*t)

plt.plot(t, x, label='displacement')
plt.axhline(A, linestyle='--')
plt.axhline(-A, linestyle='--')
plt.xlabel('time')
plt.ylabel('x(t)')
plt.title('Free undamped motion')
plt.legend()
plt.show()

energy = 0.5*m*v**2+0.5*k*x**2
plt.plot(t, energy)
plt.xlabel('time')
plt.ylabel('mechanical energy')
plt.title('Energy conservation')
plt.show()
print('maximum energy drift =', np.max(np.abs(energy-energy[0])))

### Interactive initial conditions

Changing initial displacement and velocity changes amplitude and phase, but not the natural frequency.

In [ ]:
def undamped_explorer(m=2.0, k=18.0, x0=0.4, v0=-0.9):
    omega = np.sqrt(k/m)
    A = np.sqrt(x0**2+(v0/omega)**2)
    delta = np.arctan2(v0/omega, x0)
    T = 2*np.pi/omega
    t = np.linspace(0, 4*T, 700)
    x = x0*np.cos(omega*t)+(v0/omega)*np.sin(omega*t)
    v = -x0*omega*np.sin(omega*t)+v0*np.cos(omega*t)

    plt.plot(t, x)
    plt.axhline(A, linestyle='--')
    plt.axhline(-A, linestyle='--')
    plt.xlabel('time')
    plt.ylabel('x(t)')
    plt.title('Undamped oscillator')
    plt.show()

    plt.plot(x, v)
    plt.xlabel('x')
    plt.ylabel("x'")
    plt.title('Phase portrait')
    plt.show()
    print('natural frequency =', omega)
    print('period =', T)
    print('amplitude =', A)
    print('phase =', delta)

if WIDGETS_AVAILABLE:
    interact(
        undamped_explorer,
        m=FloatSlider(min=0.5,max=6,step=0.25,value=2),
        k=FloatSlider(min=1,max=40,step=1,value=18),
        x0=FloatSlider(min=-2,max=2,step=0.1,value=0.4),
        v0=FloatSlider(min=-5,max=5,step=0.1,value=-0.9)
    )
else:
    undamped_explorer()

## Optional extension — Two springs

For two ideal springs:

- parallel:
  $$
  k_{\mathrm{eff}}=k_1+k_2;
  $$
- series:
  $$
  \frac1{k_{\mathrm{eff}}}=\frac1{k_1}+\frac1{k_2}.
  $$

The parallel combination is stiffer and therefore has a larger natural frequency.

In [ ]:
def spring_combination(k1=8.0, k2=12.0, m=2.0):
    kp = k1+k2
    ks = k1*k2/(k1+k2)
    wp = np.sqrt(kp/m)
    ws = np.sqrt(ks/m)
    t = np.linspace(0, 20, 1000)
    plt.plot(t, np.cos(wp*t), label='parallel')
    plt.plot(t, np.cos(ws*t), label='series')
    plt.xlabel('time')
    plt.ylabel('normalized displacement')
    plt.title('Effective stiffness changes the frequency')
    plt.legend()
    plt.show()
    print('parallel stiffness/frequency:', kp, wp)
    print('series stiffness/frequency:', ks, ws)

if WIDGETS_AVAILABLE:
    interact(
        spring_combination,
        k1=FloatSlider(min=1,max=30,step=1,value=8),
        k2=FloatSlider(min=1,max=30,step=1,value=12),
        m=FloatSlider(min=0.5,max=8,step=0.5,value=2)
    )
else:
    spring_combination()

## 3. Free damped motion

The equation

$$
mx''+cx'+kx=0
$$

has characteristic equation

$$
mr^2+cr+k=0.
$$

The critical damping value is

$$
c_{\mathrm{crit}}=2\sqrt{mk}.
$$

Critical damping is the boundary between oscillatory and nonoscillatory decay.

In [ ]:
def damped_explorer(m=1.0, k=9.0, c=2.0, x0=1.0, v0=0.0, final_time=12):
    disc = c**2-4*m*k
    def accel(t, x, v):
        return -(c*v+k*x)/m
    sol = solve_second_order(accel, (0, final_time), [x0,v0], rtol=1e-9, atol=1e-11)
    label = 'overdamped' if disc>1e-10 else ('critically damped' if abs(disc)<=1e-10 else 'underdamped')
    plt.plot(sol.t, sol.y[0])
    plt.axhline(0, linestyle='--')
    plt.xlabel('time')
    plt.ylabel('x(t)')
    plt.title(label)
    plt.show()
    plt.plot(sol.y[0], sol.y[1])
    plt.xlabel('x')
    plt.ylabel("x'")
    plt.title('Phase portrait')
    plt.show()
    print('discriminant =', disc)
    print('critical damping =', 2*np.sqrt(m*k))

if WIDGETS_AVAILABLE:
    interact(
        damped_explorer,
        m=FloatSlider(min=0.5,max=5,step=0.25,value=1),
        k=FloatSlider(min=1,max=25,step=1,value=9),
        c=FloatSlider(min=0,max=15,step=0.25,value=2),
        x0=FloatSlider(min=-2,max=2,step=0.1,value=1),
        v0=FloatSlider(min=-4,max=4,step=0.1,value=0),
        final_time=IntSlider(min=5,max=30,step=1,value=12)
    )
else:
    damped_explorer()

### Energy dissipation

For the damped system,

$$
E=\frac12m(x')^2+\frac12kx^2
$$

satisfies

$$
E'=-c(x')^2\le0.
$$

Thus damping removes mechanical energy.

In [ ]:
m, k, c = 1.0, 9.0, 2.5
def accel(t, x, v): return -(c*v+k*x)/m
sol = solve_second_order(accel, (0, 12), [1,0], rtol=1e-10, atol=1e-12)
E = 0.5*m*sol.y[1]**2+0.5*k*sol.y[0]**2
plt.plot(sol.t, E)
plt.xlabel('time')
plt.ylabel('energy')
plt.title('Damping produces monotone energy loss')
plt.show()

## 4. Driven motion and frequency response

For periodic forcing,

$$
mx''+cx'+kx=F_0\cos(\gamma t),
$$

the total response is

$$
x=x_c+x_p.
$$

When $c>0$, the complementary response $x_c$ decays; the periodic particular response $x_p$ remains.

In [ ]:
def forced_explorer(m=1.0, k=16.0, c=1.2, F0=5.0, gamma=3.5, x0=0.5, v0=0.0):
    def accel(t, x, v):
        return (F0*np.cos(gamma*t)-c*v-k*x)/m
    sol = solve_second_order(accel, (0, 35), [x0,v0], points=1800, rtol=1e-9, atol=1e-11)
    A = F0/np.sqrt((k-m*gamma**2)**2+(c*gamma)**2)
    phase = np.arctan2(c*gamma, k-m*gamma**2)
    xp = A*np.cos(gamma*sol.t-phase)
    plt.plot(sol.t, sol.y[0], label='total response')
    plt.plot(sol.t, xp, linestyle='--', label='steady-state response')
    plt.xlabel('time')
    plt.ylabel('x(t)')
    plt.title('Transient plus steady state')
    plt.legend()
    plt.show()
    print('steady-state amplitude =', A)
    print('phase lag =', phase)

if WIDGETS_AVAILABLE:
    interact(
        forced_explorer,
        m=FloatSlider(min=0.5,max=4,step=0.25,value=1),
        k=FloatSlider(min=2,max=30,step=1,value=16),
        c=FloatSlider(min=0.1,max=8,step=0.1,value=1.2),
        F0=FloatSlider(min=0.5,max=12,step=0.5,value=5),
        gamma=FloatSlider(min=0.2,max=8,step=0.1,value=3.5),
        x0=FloatSlider(min=-2,max=2,step=0.1,value=0.5),
        v0=FloatSlider(min=-3,max=3,step=0.1,value=0)
    )
else:
    forced_explorer()

### Resonance curve

With damping, the amplitude remains finite. Smaller damping produces a taller and narrower frequency-response peak.

In [ ]:
def resonance_curve(m=1.0, k=16.0, c=1.2, F0=5.0):
    gamma = np.linspace(0.01, 10, 1200)
    A = F0/np.sqrt((k-m*gamma**2)**2+(c*gamma)**2)
    plt.plot(gamma, A)
    plt.axvline(np.sqrt(k/m), linestyle='--', label='natural frequency')
    plt.xlabel('forcing frequency')
    plt.ylabel('steady-state amplitude')
    plt.title('Frequency response')
    plt.legend()
    plt.show()
    print('maximum displayed amplitude =', A.max())
    print('frequency at maximum =', gamma[np.argmax(A)])

if WIDGETS_AVAILABLE:
    interact(
        resonance_curve,
        m=FloatSlider(min=0.5,max=4,step=0.25,value=1),
        k=FloatSlider(min=2,max=30,step=1,value=16),
        c=FloatSlider(min=0.05,max=8,step=0.05,value=1.2),
        F0=FloatSlider(min=0.5,max=12,step=0.5,value=5)
    )
else:
    resonance_curve()

## Optional extension — LRC-series circuits

The circuit equation

$$
Lq''+Rq'+\frac1Cq=E_0\sin(\gamma t)
$$

has steady-state current amplitude

$$
I_0=\frac{E_0}{Z},
$$

where

$$
Z=\sqrt{R^2+\left(L\gamma-\frac1{C\gamma}\right)^2}
$$

is the impedance.

In [ ]:
def lrc_frequency_response(L=0.5, R=4.0, C=0.05, E0=10.0):
    gamma = np.linspace(0.1, 20, 1200)
    reactance = L*gamma-1/(C*gamma)
    Z = np.sqrt(R**2+reactance**2)
    Iamp = E0/Z
    plt.plot(gamma, Iamp)
    plt.axvline(1/np.sqrt(L*C), linestyle='--', label='undamped resonant frequency')
    plt.xlabel('angular frequency')
    plt.ylabel('current amplitude')
    plt.title('LRC steady-state current response')
    plt.legend()
    plt.show()
    print('minimum impedance =', Z.min())

if WIDGETS_AVAILABLE:
    interact(
        lrc_frequency_response,
        L=FloatSlider(min=0.1,max=2,step=0.1,value=0.5),
        R=FloatSlider(min=0.2,max=15,step=0.2,value=4),
        C=FloatSlider(min=0.01,max=0.2,step=0.01,value=0.05),
        E0=FloatSlider(min=1,max=30,step=1,value=10)
    )
else:
    lrc_frequency_response()

## Classroom Checkpoint — Exit Check

A spring–mass system has $m=2$, $k=18$, and damping $c=12$.

1. Compute $c_{\mathrm{crit}}$.
2. Classify the motion.

> Pause here. Let students commit to an answer before running the next cell.